# 04 — Gradual drift, abrupt shift, and seasonal noise

Teams say "drift" for three different mechanisms:

| Mechanism | Example in LedgerRoute | Typical response |
|-----------|------------------------|------------------|
| Gradual covariate drift | Rising online channel share | Segment eval, scheduled retrain |
| Abrupt concept/policy shift | Label rule change on day 70 | Rollback, threshold freeze, human gate |
| Seasonal noise | Sinusoidal label-rate fluctuation | Longer windows, same-week-last-year baseline |

This notebook compares **detectors and runbooks**, not a single alarm type.


In [ ]:
# From repo root: pip install -e ".[dev]"
%matplotlib inline

import numpy as np
import pandas as pd

from drift_lab import StreamConfig, generate_stream
from drift_lab.analysis import daily_channel_mix, daily_label_rate
from drift_lab.detectors import cusum_detect
from drift_lab.viz import shift_overlay_figure

cfg = StreamConfig()
cov = generate_stream("covariate_gradual", cfg)
concept = generate_stream("concept_abrupt", cfg)
noise = generate_stream("noise_only", cfg)


def normalize(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-8)


gradual = normalize(daily_channel_mix(cov).values)
abrupt = normalize(daily_label_rate(concept).values)
seasonal = normalize(daily_label_rate(noise).values)


## Overlay normalized daily signals


In [ ]:
fig = shift_overlay_figure(
    {
        "Gradual (channel mix)": gradual,
        "Abrupt (label rate)": abrupt,
        "Seasonal noise": seasonal,
    }
)


## CUSUM alarms (same threshold, different physics)


In [ ]:
rows = []
for name, series in [
    ("gradual mix", gradual),
    ("abrupt label rate", abrupt),
    ("seasonal noise", seasonal),
]:
    alarm = cusum_detect(series, threshold=4.0)
    rows.append({"signal": name, "first_alarm_day": alarm})
pd.DataFrame(rows)


## Suggested runbook mapping


In [ ]:
pd.DataFrame(
    [
        ("Gradual covariate", "Segment accuracy + mix plot", "Scheduled retrain / reweight"),
        ("Abrupt concept", "ECE + queue volume", "Policy rollback, threshold freeze"),
        ("Seasonal noise", "Same ISO week last year", "Extend window; no retrain"),
    ],
    columns=["mechanism", "confirm with", "first response"],
)


## Takeaway

- **Do not share one on-call runbook** for all alarm types.
- Control **multiplicity** when testing many features hourly—digest sub-threshold moves.
- **Replay** a past week offline before retraining on noise.
